# Final Project Science Fair Demo

## Install Package

In [ ]:
mamba create -n cx-demo python=3.12 -y && mamba activate cx-demo

In [ ]:
pip install \
  --index-url https://test.pypi.org/simple/ \
  --extra-index-url https://pypi.org/simple/ \
  clinicalxai

pip install skl2onnx # dev package requirement

## Loading Data and Model

### A. If you already have an ONNX Model:

In [5]:
from pathlib import Path
from clinicalxai.model import OnnxModel

model_path = "./diabetes_clf.onnx"
model = OnnxModel(model_path)
model

### B. If you have a trained model and no .onnx file

In [10]:
from clinicalxai.datasets import load_diabetes_dataset
from sklearn.linear_model import LogisticRegression
from skl2onnx.common.data_types import FloatTensorType # you will need to install skl2onnx package separately !!!
from skl2onnx import convert_sklearn

In [15]:
X_train, X_test, y_train, y_test = load_diabetes_dataset()

clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)
initial_types = [("input", FloatTensorType([None, X_train.shape[1]]))]

onnx_model = convert_sklearn(
    clf, initial_types=initial_types, options={id(clf): {"zipmap": False}}
)

model_path = Path("./diabetes_clf.onnx")
model_path.write_bytes(onnx_model.SerializeToString())

trained_model = OnnxModel(model_path)

## Create ClassifierExplainer Object and Create XAI Report

In [16]:
from clinicalxai.generate_report import render_report
from clinicalxai.explainers.classifier import ClassifierExplainer

# Create explainer object
explainer = ClassifierExplainer(
    trained_model, X_test, y_test, labels=["No Diabetes", "Diabetes"]
)

# Export XAI HTML
render_report(
    explainer,
    output_path="./clinicalxai-demo-report.html",
    title="Diabetes Prediction XAI Report",
)

/Users/allyatkins/miniforge3/envs/cx-test/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PermutationExplainer explainer: 50737it [02:18, 343.79it/s]                           


Report generated and saved to: clinicalxai-demo-report.html


'./clinicalxai-demo-report.html'